# Indonesia — Ookla Open Data, Province-Quarterly Aggregation

Builds `data/exports/ookla_indonesia_province_quarterly.csv` and
`data/exports/ookla_mobile_indonesia_province_quarterly.csv` in the **exact same format** as the
other 8 countries (see `notebooks/ookla/malaysia_eda.ipynb` for the pipeline this mirrors).

Indonesia joins the paper as a **9th country** (decision made 2026-07-30) alongside the original
eight (Thailand, Vietnam, Philippines, Singapore, Cambodia, Laos, Malaysia, Myanmar) — not a swap.

Uses the same globally-cached raw Ookla tiles already in `data/ookla/raw/` (12 quarters, fixed +
mobile, all countries pulled from this same global release via per-country bounding-box filtering)
— no new download needed for Ookla specifically.

Reference data (`data/reference/indonesia_reference.csv`, `data/geo/indonesia_provinces.geojson`)
already exists — see `data/reference/indonesia_reference_PROVENANCE.md` for sourcing and caveats
(notably: `pop_2024` is actually mid-2025 data, and Indonesia's 38 provinces are folded to 34 per
GADM 4.1 2022 boundaries, pre-Papua-split — must be disclosed in the paper wherever this is used).


In [1]:
import glob, os, warnings
import geopandas as gpd
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

DATASET_DIR = '../../data/ookla/raw'
GEOJSON     = '../../data/geo/indonesia_provinces.geojson'
PROV_REF    = '../../data/reference/indonesia_reference.csv'

# Ookla S3 filename convention: {YEAR}-{MM}-01_performance_{fixed,mobile}_tiles.parquet
Q_TO_MONTH = {1: 1, 2: 4, 3: 7, 4: 10}

indonesia_gdf = gpd.read_file(GEOJSON).to_crs(4326)
state_bounds = indonesia_gdf.total_bounds
bbox_filters = [
    ('tile_y', '<=', state_bounds[3]), ('tile_y', '>=', state_bounds[1]),
    ('tile_x', '<=', state_bounds[2]), ('tile_x', '>=', state_bounds[0]),
]
print(f"Indonesia bounding box: {state_bounds}")
print(f"Provinces in geojson: {len(indonesia_gdf)}")


Indonesia bounding box: [ 95.0109845 -11.0075889 141.0194444   5.9068968]
Provinces in geojson: 34


## Reusable aggregation function

Same logic as every other country's notebook: bbox-filter the global raw tile file, spatial-join
to province polygons, test-count-weighted average per province, per quarter.

In [2]:
COLS = ['tile_x','tile_y','tests','devices','avg_d_kbps','avg_u_kbps','avg_lat_ms']

def build_master(network_type: str) -> pd.DataFrame:
    """network_type: 'fixed' or 'mobile'"""
    records = []
    files = sorted(glob.glob(os.path.join(DATASET_DIR, f'*{network_type}_tiles*.parquet')))
    assert len(files) == 12, f"expected 12 quarterly files, found {len(files)}"

    for fp in files:
        fname   = os.path.basename(fp)
        year    = int(fname[:4])
        quarter = int(fname[6])
        month   = Q_TO_MONTH[quarter]
        period_start = pd.Timestamp(year=year, month=month, day=1)
        label   = f"{year}-Q{quarter}"

        raw = pd.read_parquet(fp, columns=COLS, filters=bbox_filters)

        tiles_gdf = gpd.GeoDataFrame(
            raw,
            geometry=gpd.points_from_xy(raw.tile_x, raw.tile_y),
            crs='EPSG:4326'
        ).drop(columns=['tile_x', 'tile_y'])

        joined = gpd.sjoin(tiles_gdf, indonesia_gdf[['name', 'geometry']], how='inner', predicate='intersects')

        agg = (
            joined.groupby('name')
            .apply(lambda g: pd.Series({
                'avg_d_kbps_wt': np.average(g['avg_d_kbps'], weights=g['tests']),
                'avg_u_kbps_wt': np.average(g['avg_u_kbps'], weights=g['tests']),
                'avg_lat_ms_wt': np.average(g['avg_lat_ms'], weights=g['tests']),
                'total_tests'  : g['tests'].sum(),
                'total_devices': g['devices'].sum(),
                'n_tiles'      : len(g),
            }), include_groups=False)
            .reset_index()
        )
        agg['year'] = year
        agg['quarter'] = quarter
        agg['month'] = month
        agg['period_start'] = period_start
        agg['label'] = label
        records.append(agg)
        print(f"  [{network_type}] {label} ({period_start.date()}): {len(raw):>8,} raw tiles -> {len(agg)} provinces")

    master = pd.concat(records, ignore_index=True)
    master['avg_d_mbps'] = master['avg_d_kbps_wt'] / 1000
    master['avg_u_mbps'] = master['avg_u_kbps_wt'] / 1000
    master.drop(columns=['avg_d_kbps_wt', 'avg_u_kbps_wt'], inplace=True)

    prov_ref = pd.read_csv(PROV_REF)
    master = master.merge(prov_ref.rename(columns={'province_en': 'name'}), on='name', how='left')

    master['is_reliable'] = (
        (master['total_tests'] >= 100) &
        (master['n_tiles'] >= 5)
    )
    return master


## Fixed broadband

In [3]:
master_fixed = build_master('fixed')
print(f"\nMaster shape: {master_fixed.shape}")
print(f"Provinces: {master_fixed['name'].nunique()} / {len(indonesia_gdf)} in geojson")
reliable_count = master_fixed.groupby('label')['is_reliable'].sum()
print(f"Overall reliable: {master_fixed['is_reliable'].sum()} / {len(master_fixed)} ({master_fixed['is_reliable'].mean():.1%})")


  [fixed] 2023-Q1 (2023-01-01):  187,064 raw tiles -> 34 provinces


  [fixed] 2023-Q2 (2023-04-01):  194,899 raw tiles -> 34 provinces


  [fixed] 2023-Q3 (2023-07-01):  200,739 raw tiles -> 34 provinces


  [fixed] 2023-Q4 (2023-10-01):  209,518 raw tiles -> 34 provinces


  [fixed] 2024-Q1 (2024-01-01):  218,168 raw tiles -> 34 provinces


...

  [fixed] 2024-Q4 (2024-10-01):  228,242 raw tiles -> 34 provinces


  [fixed] 2025-Q1 (2025-01-01):  235,522 raw tiles -> 34 provinces


  [fixed] 2025-Q2 (2025-04-01):  257,796 raw tiles -> 34 provinces


  [fixed] 2025-Q3 (2025-07-01):  271,883 raw tiles -> 34 provinces


  [fixed] 2025-Q4 (2025-10-01):  284,199 raw tiles -> 34 provinces

Master shape: (408, 22)
Provinces: 34 / 34 in geojson
Overall reliable: 408 / 408 (100.0%)


In [4]:
# Coverage check -- same sanity check as every other country's notebook
expected_n = len(pd.read_csv(PROV_REF))
expected = expected_n * master_fixed['label'].nunique()
print(f"Expected rows: {expected} | Actual: {len(master_fixed)} | Missing: {expected - len(master_fixed)}")

all_provinces = set(indonesia_gdf['name'])
covered = set(master_fixed['name'].unique())
if all_provinces - covered:
    print(f"Provinces with zero coverage in ALL quarters: {all_provinces - covered}")
else:
    print("All provinces have at least some coverage.")

low_test = master_fixed[master_fixed['total_tests'] < 100][['name','label','total_tests','n_tiles']].sort_values('total_tests')
print(f"\nLow-test quarters (< 100 tests): {len(low_test)} cases")
if len(low_test):
    print(low_test.to_string(index=False))


Expected rows: 408 | Actual: 408 | Missing: 0
All provinces have at least some coverage.

Low-test quarters (< 100 tests): 0 cases


In [5]:
# Drop unreliable rows before export -- matches every other country's export convention
master_fixed_reliable = master_fixed[master_fixed['is_reliable']].copy()
print(f"Filtered to reliable province-quarters: {len(master_fixed_reliable)} rows kept (of {len(master_fixed)})")


Filtered to reliable province-quarters: 408 rows kept (of 408)


In [6]:
# Export -- same EXPORT_COLS/rename convention as every other country
os.makedirs("../../data/exports", exist_ok=True)

EXPORT_COLS = [
    "name", "label", "year", "quarter",
    "avg_d_mbps", "avg_u_mbps", "avg_lat_ms_wt",
    "total_tests", "n_tiles", "is_reliable",
    "region", "internet_tier", "pop_2024",
    "gdp_per_capita_thb_2021", "density_per_km2",
]
existing_cols = [c for c in EXPORT_COLS if c in master_fixed_reliable.columns]
out = master_fixed_reliable[existing_cols].copy()
out = out.rename(columns={"name": "province", "label": "quarter"})

PATH = "../../data/exports/ookla_indonesia_province_quarterly.csv"
out.to_csv(PATH, index=False)
print(f"Exported {len(out)} rows -> {PATH}")
out.head(3)


Exported 408 rows -> ../../data/exports/ookla_indonesia_province_quarterly.csv


,province,quarter,year,quarter,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_thb_2021,density_per_km2
0,Aceh,2023-Q1,2023,1,24.272803,12.047070,20.520354,17515.0,1990.0,True,Sumatra,4,5626000,233536.0,99
1,Bali,2023-Q1,2023,1,36.880704,35.881273,12.118241,184192.0,4274.0,True,Lesser Sunda Islands,3,4461300,339360.0,799
2,Bangka Belitung,2023-Q1,2023,1,29.360677,19.791669,18.646611,6964.0,659.0,True,Sumatra,2,1550800,392960.0,93


## Mobile

In [7]:
master_mobile = build_master('mobile')
print(f"\nMaster shape: {master_mobile.shape}")
reliable_count = master_mobile.groupby('label')['is_reliable'].sum()
print(f"Overall reliable: {master_mobile['is_reliable'].sum()} / {len(master_mobile)} ({master_mobile['is_reliable'].mean():.1%})")


  [mobile] 2023-Q1 (2023-01-01):  207,993 raw tiles -> 34 provinces


  [mobile] 2023-Q2 (2023-04-01):  216,459 raw tiles -> 34 provinces


  [mobile] 2023-Q3 (2023-07-01):  212,883 raw tiles -> 34 provinces


  [mobile] 2023-Q4 (2023-10-01):  210,193 raw tiles -> 34 provinces


  [mobile] 2024-Q1 (2024-01-01):  205,446 raw tiles -> 34 provinces


...

  [mobile] 2024-Q4 (2024-10-01):  210,668 raw tiles -> 34 provinces


  [mobile] 2025-Q1 (2025-01-01):  200,815 raw tiles -> 34 provinces


  [mobile] 2025-Q2 (2025-04-01):  206,393 raw tiles -> 34 provinces


  [mobile] 2025-Q3 (2025-07-01):  214,309 raw tiles -> 34 provinces


  [mobile] 2025-Q4 (2025-10-01):  228,646 raw tiles -> 34 provinces

Master shape: (408, 22)
Overall reliable: 408 / 408 (100.0%)


In [8]:
master_mobile_reliable = master_mobile[master_mobile['is_reliable']].copy()
print(f"Filtered to reliable province-quarters: {len(master_mobile_reliable)} rows kept (of {len(master_mobile)})")

existing_cols = [c for c in EXPORT_COLS if c in master_mobile_reliable.columns]
out = master_mobile_reliable[existing_cols].copy()
out = out.rename(columns={"name": "province", "label": "quarter"})

PATH = "../../data/exports/ookla_mobile_indonesia_province_quarterly.csv"
out.to_csv(PATH, index=False)
print(f"Exported {len(out)} rows -> {PATH}")
out.head(3)


Filtered to reliable province-quarters: 408 rows kept (of 408)
Exported 408 rows -> ../../data/exports/ookla_mobile_indonesia_province_quarterly.csv


,province,quarter,year,quarter,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_thb_2021,density_per_km2
0,Aceh,2023-Q1,2023,1,29.848491,12.777939,46.376614,18507.0,3078.0,True,Sumatra,4,5626000,233536.0,99
1,Bali,2023-Q1,2023,1,34.330959,19.048167,35.354932,52396.0,3447.0,True,Lesser Sunda Islands,3,4461300,339360.0,799
2,Bangka Belitung,2023-Q1,2023,1,32.597127,21.747353,45.210907,8985.0,1281.0,True,Sumatra,2,1550800,392960.0,93


## Sanity check against an existing country's export

Confirms schema match, byte-for-byte column names, before this gets pulled into the cross-country
notebooks (`rq_stats_crosscountry.ipynb`, `rq1_thresholds_ookla.ipynb`, `rq2_trends.ipynb`).

In [9]:
ref_df = pd.read_csv('../../data/exports/ookla_malaysia_province_quarterly.csv')
new_df = pd.read_csv('../../data/exports/ookla_indonesia_province_quarterly.csv')
assert list(ref_df.columns) == list(new_df.columns), "COLUMN SCHEMA MISMATCH -- fix before using downstream"
print("Schema matches ookla_malaysia_province_quarterly.csv exactly.")
print(f"Indonesia fixed: {len(new_df)} rows, {new_df['province'].nunique()} provinces, "
      f"{new_df['quarter'].nunique()} quarters")


Schema matches ookla_malaysia_province_quarterly.csv exactly.
Indonesia fixed: 408 rows, 34 provinces, 12 quarters


## Next steps

- Add Indonesia to `COUNTRY_FILES` in `notebooks/comparison/rq_stats_crosscountry.ipynb`, `rq2_trends.ipynb`,
  and any RQ1 threshold notebook -- re-run those with 9 countries instead of 8.
- Update every "eight countries" / "8 countries" claim in `docs/paper_draft.md` (Abstract, Contributions,
  Table 1, Results tables, Limitations, Conclusion) to nine, once these numbers are in.
- Disclose in Methodology: Indonesia's `pop_2024` is actually mid-2025 data, and its 34 provinces
  (GADM 4.1, 2022 boundaries) predate the 38-province split following Papua's 2022-24 subdivision --
  same kind of boundary-year caveat already disclosed for Myanmar (Section 2.1.3).
